In [ ]:
# ! pip install -Uq openai dotenv

In [1]:
import mlflow
import openai
import os
from dotenv import load_dotenv
from config import MODEL_NAME, PROMPT_NAME, PROMPT_VERSION, REASONING, MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME

load_dotenv()

d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


True

In [2]:
system_prompt = """You are a Vision Language Model designed to extract structured data from invoice receipts.
Task:
Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

Requirements:
1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
2. Preserve exact formatting for all the extracted values.  
3. Do not output fields that lack data—omit empty keys.  
4. Do not add any information not present in the invoice.
5. In case of prices and currencies, ensure to maintain the original format without any modifications.

Schema:
{schema}

Output:
Return valid, minimal JSON matching this schema - no extraneous keys or null values.
"""

print(system_prompt)

You are a Vision Language Model designed to extract structured data from invoice receipts.
Task:
Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

Requirements:
1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
2. Preserve exact formatting for all the extracted values.  
3. Do not output fields that lack data—omit empty keys.  
4. Do not add any information not present in the invoice.
5. In case of prices and currencies, ensure to maintain the original format without any modifications.

Schema:
{schema}

Output:
Return valid, minimal JSON matching this schema - no extraneous keys or null values.



In [3]:
prompt = mlflow.genai.register_prompt(
    name = PROMPT_NAME,
    template = system_prompt
)

prompt

2025/08/16 16:51:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: invoice-extraction-gpt-5-mini-prompt, version 3


PromptVersion(name=invoice-extraction-gpt-5-mini-prompt, version=3, template=You are a Vision Language Mode...)

In [4]:
print(mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{PROMPT_VERSION}").template)

You are a Vision Language Model designed to extract structured data from invoice receipts.
Task:
Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

Requirements:
1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
2. Preserve exact formatting for all the extracted values.  
3. Do not output fields that lack data—omit empty keys.  
4. Do not add any information not present in the invoice.
5. In case of prices and currencies, ensure to maintain the original format without any modifications.

Schema:
{schema}

Output:
Return valid, minimal JSON matching this schema - no extraneous keys or null values.



In [5]:
def log_invoice_extraction_model():
    system_prompt = mlflow.load_prompt(f"prompts:/{PROMPT_NAME}/{PROMPT_VERSION}").template
    print("System Prompt: ", system_prompt)
    with mlflow.start_run(run_name=f"{MODEL_NAME}-{REASONING}") as run:
        model_info = mlflow.openai.log_model(
            model=MODEL_NAME,
            reasoning={
                "effort": REASONING,
            },
            task=openai.chat.completions,
            name=f"{MODEL_NAME}-{REASONING}",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": system_prompt},
                        {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,{image_base64}"}},
                    ],
                }
            ],
        )

    return model_info.model_uri

In [6]:
log_invoice_extraction_model()

C:\Users\shrin\AppData\Local\Temp\ipykernel_26564\2733285870.py:2: FutureWarning: The `mlflow.load_prompt` API is moved to the `mlflow.genai` namespace. Please use `mlflow.genai.load_prompt` instead. The original API will be removed in the future release.
  system_prompt = mlflow.load_prompt(f"prompts:/{PROMPT_NAME}/{PROMPT_VERSION}").template


System Prompt:  You are a Vision Language Model designed to extract structured data from invoice receipts.
Task:
Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

Requirements:
1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
2. Preserve exact formatting for all the extracted values.  
3. Do not output fields that lack data—omit empty keys.  
4. Do not add any information not present in the invoice.
5. In case of prices and currencies, ensure to maintain the original format without any modifications.

Schema:
{schema}

Output:
Return valid, minimal JSON matching this schema - no extraneous keys or null values.

🏃 View run gpt-5-mini-low at: http://localhost:8080/#/experiments/341604693180857072/runs/2d6e3472bad34ab99f9c7171c7cc5e31
🧪 View experiment at: http://localhost:8080/#/experiments/341604693180857072


'models:/m-ef950965995a451b8e48b4a6c87e82af'